# XGBoost Crop Recommendation Prototype

This notebook trains a six-feature prototype using N, P, K, pH, temperature, and rainfall.

Soil type and soil moisture are not used by this model because their source units and training signal are not approved. The Test split remains sealed. All reported results use Validation.


## 1 Setup


In [ ]:
from pathlib import Path
import json
import platform
import sys
import time

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.modeling import (
    ARTIFACT_DIR,
    FEATURES,
    RANDOM_STATE,
    TARGET,
    encode_training_validation_targets,
    evaluate_probabilities,
    load_training_validation_splits,
    plot_evaluation_dashboard,
    prediction_table,
    save_evaluation_outputs,
    training_validation_summary,
)

np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
print("Python", platform.python_version())
print("Project root", PROJECT_ROOT)


In [ ]:
try:
    import xgboost as xgb
except Exception as error:
    raise RuntimeError(
        "XGBoost is unavailable. Install xgboost and the macOS libomp runtime before running."
    ) from error
print("XGBoost", xgb.__version__)


## 2 Frozen Data Split


In [ ]:
train, validation, sealed_test_summary, split_integrity = load_training_validation_splits()
label_encoder, y_train, y_validation = encode_training_validation_targets(train, validation)

X_train = train[FEATURES].copy()
X_validation = validation[FEATURES].copy()

display(training_validation_summary(train, validation))
print("Features", FEATURES)
print("Crop classes", list(label_encoder.classes_))
print("Sealed Test metadata", sealed_test_summary)


In [ ]:
distribution = pd.concat(
    [
        train.assign(data_split="Train"),
        validation.assign(data_split="Validation"),
    ],
    ignore_index=True,
)
class_counts = (
    distribution.groupby([TARGET, "data_split"])
    .size()
    .rename("row_count")
    .reset_index()
)

plt.figure(figsize=(14, 7))
sns.barplot(data=class_counts, x=TARGET, y="row_count", hue="data_split")
plt.title("Frozen Split Class Distribution")
plt.xlabel("Crop Label")
plt.ylabel("Row Count")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


## 3 Train XGBoost


In [ ]:
from sklearn.metrics import top_k_accuracy_score

MODEL_NAME = "xgboost"
MAX_ROUNDS = 1000
EARLY_STOPPING_ROUNDS = 60

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=FEATURES)
dvalidation = xgb.DMatrix(
    X_validation, label=y_validation, feature_names=FEATURES
)

class TqdmTrainingCallback(xgb.callback.TrainingCallback):
    def __init__(self, total_rounds):
        self.total_rounds = total_rounds
        self.progress = None

    def before_training(self, model):
        self.progress = tqdm(total=self.total_rounds, desc="XGBoost rounds", unit="round")
        return model

    def after_iteration(self, model, epoch, evals_log):
        self.progress.update(1)
        validation_loss = evals_log["validation"]["mlogloss"][-1]
        self.progress.set_postfix(loss=f"{validation_loss:.4f}")
        return False

    def after_training(self, model):
        self.progress.close()
        return model

parameters = {
    "objective": "multi:softprob",
    "num_class": len(label_encoder.classes_),
    "eval_metric": "mlogloss",
    "eta": 0.03,
    "max_depth": 5,
    "min_child_weight": 1,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "lambda": 1.0,
    "alpha": 0.0,
    "seed": RANDOM_STATE,
    "nthread": -1,
}
evaluation_results = {}
training_start = time.perf_counter()
model = xgb.train(
    parameters,
    dtrain,
    num_boost_round=MAX_ROUNDS,
    evals=[(dtrain, "train"), (dvalidation, "validation")],
    evals_result=evaluation_results,
    callbacks=[
        TqdmTrainingCallback(MAX_ROUNDS),
        xgb.callback.EarlyStopping(
            rounds=EARLY_STOPPING_ROUNDS,
            metric_name="mlogloss",
            data_name="validation",
            save_best=True,
        ),
    ],
    verbose_eval=False,
)
training_seconds = time.perf_counter() - training_start
best_round = int(getattr(model, "best_iteration", len(evaluation_results["validation"]["mlogloss"]) - 1)) + 1
validation_probabilities = model.predict(
    dvalidation, iteration_range=(0, best_round)
)

history_rows = []
history_progress = tqdm(
    range(10, best_round + 1, 10),
    desc="XGBoost history",
    unit="checkpoint",
)
for round_count in history_progress:
    checkpoint_probabilities = model.predict(
        dvalidation, iteration_range=(0, round_count)
    )
    checkpoint_top_3 = top_k_accuracy_score(
        y_validation,
        checkpoint_probabilities,
        k=3,
        labels=np.arange(len(label_encoder.classes_)),
    )
    history_rows.append(
        {
            "iteration": round_count,
            "training_loss": evaluation_results["train"]["mlogloss"][round_count - 1],
            "validation_loss": evaluation_results["validation"]["mlogloss"][round_count - 1],
            "validation_top_3": checkpoint_top_3,
            "elapsed_seconds": np.nan,
        }
    )
if not history_rows or history_rows[-1]["iteration"] != best_round:
    final_top_3 = top_k_accuracy_score(
        y_validation,
        validation_probabilities,
        k=3,
        labels=np.arange(len(label_encoder.classes_)),
    )
    history_rows.append(
        {
            "iteration": best_round,
            "training_loss": evaluation_results["train"]["mlogloss"][best_round - 1],
            "validation_loss": evaluation_results["validation"]["mlogloss"][best_round - 1],
            "validation_top_3": final_top_3,
            "elapsed_seconds": training_seconds,
        }
    )
training_history = pd.DataFrame(history_rows)
print("Best boosting round", best_round)
print("XGBoost training completed in", round(training_seconds, 3), "seconds")
display(training_history.tail())


## 4 Validation Evaluation


In [ ]:
validation_metrics, validation_per_class, validation_matrix, validation_calibration = (
    evaluate_probabilities(y_validation, validation_probabilities, label_encoder)
)
validation_predictions = prediction_table(
    y_validation, validation_probabilities, label_encoder
)

metrics_table = pd.DataFrame(
    {"metric": list(validation_metrics), "value": list(validation_metrics.values())}
)
display(metrics_table)
display(validation_per_class.sort_values("f1-score").reset_index(drop=True))

plot_evaluation_dashboard(
    MODEL_NAME,
    validation_metrics,
    validation_per_class,
    validation_matrix,
    validation_calibration,
    y_validation,
    validation_probabilities,
    label_encoder,
    training_history,
)


## 5 Feature Importance


In [ ]:
gain = model.get_score(importance_type="gain")
importance = pd.DataFrame(
    {"feature": FEATURES, "importance": [gain.get(feature, 0.0) for feature in FEATURES]}
).sort_values("importance", ascending=True)

plt.figure(figsize=(9, 5))
sns.barplot(data=importance, x="importance", y="feature", color="#2f7d32")
plt.title("XGBoost Feature Importance by Gain")
plt.xlabel("Gain")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


## 6 Save Validation Artifacts


In [ ]:
output_dir = save_evaluation_outputs(
    MODEL_NAME,
    validation_metrics,
    validation_per_class,
    validation_calibration,
    validation_predictions,
    training_history,
    training_seconds,
)
print("Saved evaluation artifacts to", output_dir)
print("Training time seconds", round(training_seconds, 3))
print("Test split used", False)


In [ ]:
model.save_model(output_dir / "model.json")
joblib.dump(label_encoder, output_dir / "label_encoder.joblib")
print("Saved model", output_dir / "model.json")
